In [2]:
# Cell 1: Import thư viện
import pandas as pd
import streamlit as st
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import re
import numpy as np
from tqdm import tqdm
from underthesea import word_tokenize
from joblib import Parallel, delayed

In [3]:
# Cell 2: Hàm tải stopwords
def get_stopwords_list(stop_file_path):
    try:
        with open(stop_file_path, 'r', encoding="utf-8") as f:
            stopwords = f.readlines()
            stop_set = set(m.strip() for m in stopwords)
            return list(frozenset(stop_set))
    except FileNotFoundError:
        print("File stopwords không tồn tại.")
        return []

In [4]:
# Cell 3: Hàm tiền xử lý văn bản
def preprocess_text(data, stopwords):
    data = data.lower()
    data = re.sub(r'\W+', ' ', data)
    data = word_tokenize(data, format="text")
    data = ' '.join([word for word in data.split() if word not in stopwords])
    return data

def preprocess_description(products_df, stopwords):
    products_df = products_df.copy()
    def enhance_description(row):
        desc = row['description']
        if pd.isna(desc) or desc.strip().lower() in ['không có mô tả', 'xem thêm'] or len(desc.split()) < 5:
            desc = f"{row['name_product']} {row['category']}"
        return preprocess_text(desc, stopwords)
    
    products_df['processed_description'] = products_df.apply(enhance_description, axis=1)
    products_df = products_df[products_df['processed_description'].str.strip() != '']
    return products_df

In [5]:
# Cell 4: Hàm tạo ma trận TF-IDF
def create_tfidf_matrix(processed_texts, max_features=4500):
    vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = vectorizer.fit_transform(processed_texts)
    return vectorizer, tfidf_matrix

In [7]:
# Cell 5: Hàm gợi ý sản phẩm
def get_content_based_recommendations(username, products_df, ratings_df, num_recommendations=5, stopwords=[], rating_threshold=4, description_matrix=None):
    if description_matrix is None:
        products_df = preprocess_description(products_df, stopwords)
        if products_df.empty:
            return []
        vectorizer, description_matrix = create_tfidf_matrix(products_df['processed_description'])
    else:
        products_df = products_df.copy()
    
    user_ratings = ratings_df[(ratings_df['username'] == username) & (ratings_df['rating'] >= rating_threshold)]
    if user_ratings.empty:
        return []
    
    user_product_indices = products_df[products_df['id_product'].isin(user_ratings['id_product'])].index
    if len(user_product_indices) == 0:
        return []
    
    user_profile = np.asarray(description_matrix[user_product_indices].mean(axis=0))
    cosine_sim = cosine_similarity(user_profile, description_matrix)[0]
    
    sim_scores = list(enumerate(cosine_sim))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    rated_products = set(user_ratings['id_product'])
    sim_scores = [score for score in sim_scores if products_df.iloc[score[0]]['id_product'] not in rated_products]
    sim_scores = sim_scores[:num_recommendations]
    product_indices = [i[0] for i in sim_scores]
    
    return products_df.iloc[product_indices][['id_product', 'name_product', 'description', 'category', 'img']].to_dict('records')

In [8]:
# Cell 6: Hàm đánh giá người dùng đơn lẻ
def evaluate_single_user(username, products_df, train_df, test_df, k, stopwords, rating_threshold, description_matrix):
    test_products = set(test_df[(test_df['username'] == username) & (test_df['rating'] >= rating_threshold)]['id_product'])
    if not test_products:
        return 0, 0, 0
    
    recommendations = get_content_based_recommendations(
        username, products_df, train_df, k, stopwords, rating_threshold, description_matrix
    )
    recommended_product_ids = [rec['id_product'] for rec in recommendations]
    
    true_positives = len(set(recommended_product_ids) & test_products)
    precision = true_positives / k if k > 0 else 0
    recall = true_positives / len(test_products) if len(test_products) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1